# Langchain

I'm using Ollama for free access, for model like OpenAI, should have an API Key (with quota to be able run the llm).

In [2]:
from langchain_community.llms import Ollama

model = Ollama(model='llama3')
model.invoke("the sky is")

'Blue!'

Instead of a single prompt string, chat models make use of different types of chat
message interfaces associated with each role mentioned previously. These include the
following:

`HumanMessage` : A message sent from the perspective of the human, with the user role 

`AIMessage` :A message sent from the perspective of the AI that the human is interacting with,
with the assistant role

`SystemMessage` : A message setting the instructions the AI should follow, with the system role

`ChatMessage` : A message allowing for arbitrary setting of role

In [5]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage

model = Ollama(model='llama3')
prompt = [HumanMessage("What is the capital of France?")]
model.invoke(prompt)

'The capital of France is Paris!'

In [6]:
from langchain_core.messages import SystemMessage

system_msg = SystemMessage(
'''You are a helpful assistant that responds to questions with three
exclamation marks.'''
)
human_msg = HumanMessage('What is the capital of France?')
model.invoke([system_msg, human_msg])

'!!! Paris!!!'

## Making LLM Prompts Reusable

The previous section showed how the prompt instruction significantly influences the
model’s output. Prompts help the model understand context and generate relevant
answers to queries.

Here is an example of a detailed prompt:

`Answer the question based on the context below. If the question cannot be
answered using the information provided, answer with "I don't know".
Context: The most recent advancements in NLP are being driven by Large Language
Models (LLMs). These models outperform their smaller counterparts and have
become invaluable for developers who are creating applications with NLP
capabilities. Developers can tap into these models through Hugging Face's
`transformers` library, or by utilizing OpenAI and Cohere's offerings through
the `openai` and `cohere` libraries, respectively.`

``Question: Which model providers offer LLMs?``

``Answer:``

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)

template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

StringPromptValue(text='Answer the question based on the\ncontext below. If the question cannot be answered using the information\nprovided, answer with "I don\'t know".\nContext: The most recent advancements in NLP are being driven by Large\nLanguage Models (LLMs). These models outperform their smaller\ncounterparts and have become invaluable for developers who are creating\napplications with NLP capabilities. Developers can tap into these\nmodels through Hugging Face\'s `transformers` library, or by utilizing\nOpenAI and Cohere\'s offerings through the `openai` and `cohere`\nlibraries, respectively.\nQuestion: Which model providers offer LLMs?\nAnswer: ')

In [ ]:
template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)

prompt = template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})
completion = model.invoke(prompt)

In [10]:
completion

'Hugging Face, OpenAI, and Cohere.'

In [11]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
('system', '''Answer the question based on the context below. If the
question cannot be answered using the information provided, answer with
"I don\'t know".'''),
('human', 'Context: {context}'),
('human', 'Question: {question}'),
])

template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

ChatPromptValue(messages=[SystemMessage(content='Answer the question based on the context below. If the\nquestion cannot be answered using the information provided, answer with\n"I don\'t know".', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context: The most recent advancements in NLP are being driven by Large\nLanguage Models (LLMs). These models outperform their smaller\ncounterparts and have become invaluable for developers who are creating\napplications with NLP capabilities. Developers can tap into these\nmodels through Hugging Face's `transformers` library, or by utilizing\nOpenAI and Cohere's offerings through the `openai` and `cohere`\nlibraries, respectively.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Question: Which model providers offer LLMs?', additional_kwargs={}, response_metadata={})])

In [13]:

template = ChatPromptTemplate.from_messages([
('system', '''Answer the question based on the context below. If the
question cannot be answered using the information provided, answer with
"I don\'t know".'''),
('human', 'Context: {context}'),
('human', 'Question: {question}'),
])

prompt = template.invoke({
"context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
"question": "Which model providers offer LLMs?"
})

model.invoke(prompt)

'According to the context, OpenAI and Cohere offer Large Language Models (LLMs) through their respective libraries, which are accessible via the `openai` and `cohere` libraries. Therefore, the answer is:\n\nOpenAI and Cohere'

## Getting Specific Formats out of LLMs

Plain text outputs are useful, but there may be use cases where you need the LLM to
generate a structured output—that is, output in a machine-readable format, such as
JSON, XML, CSV, or even in a programming language such as Python or JavaScript.
This is very useful when you intend to hand that output off to some other piece of
code, making an LLM play a part in your larger application.

In [ ]:
from langchain_ollama import ChatOllama 
from pydantic import BaseModel, Field

class AnswerWithJustification(BaseModel):
    answer: str = Field(description="The answer to the question")
    justification: str = Field(description="Justification for the answer")

llm = ChatOllama(model="llama3", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)

result = structured_llm.invoke("What weighs more, a pound of bricks or a pound of feathers?")
print(result)

answer='The same!' justification='Both weigh one pound.'


In [24]:
llm = ChatOllama(model='llama3')
completion = llm.invoke('Hi there!')
completions = llm.batch(['Hi there!', 'Bye!'])
for token in model.stream('Bye!'):
    print(token)

By
e
!
 It
 was
 nice
 chatting
 with
 you
.
 Have
 a
 great
 day
!



In this example, you see how the three main methods work:

• ``invoke()`` takes a single input and returns a single output.

• ``batch()`` takes a list of outputs and returns a list of outputs.

• ``stream()`` takes a single input and returns an iterator of parts of the output as
they become available.

## Imperative Composition

Imperative composition is just a fancy name for writing the code you’re used to
writing, composing these components into functions and classes. Here’s an example
combining prompts, models, and output parsers:

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

template = ChatPromptTemplate.from_messages([
('system', 'You are a helpful assistant.'),
('human', '{question}'),
])
model = ChatOllama(model='llama3')
# combine them in a function
# @chain decorator adds the same Runnable interface for any function you write
@chain
def chatbot(values):
    prompt = template.invoke(values)
    return model.invoke(prompt)
# use it
chatbot.invoke({"question": "Which model providers offer LLMs?"})

AIMessage(content="Several language model provider companies offer Large Language Models (LLMs). Here's a list of some well-known ones:\n\n1. **Google**: Google offers several LLM models, including BERT and its variants, like RoBERTa and ALBERT.\n2. **Microsoft**: Microsoft provides the Turing NLG model, which is an LLM designed for natural language generation tasks.\n3. **Hugging Face**: Hugging Face is a popular provider of pre-trained LLMs, offering a wide range of models, including BERT, RoBERTa, and DistilBERT. They also provide a library called Transformers for easy integration with these models.\n4. **DeepMind**: DeepMind, a subsidiary of Alphabet Inc., offers the Transformer-XL model, which is an LLM designed for long-range dependencies and generative tasks.\n5. **OpenAI**: OpenAI provides the GPT-3 model, which is a highly advanced LLM capable of generating human-like text.\n6. **Stanford NLP Group**: The Stanford NLP Group offers the BERT-based models, including SciBERT and C

## Declarative Composition

LCEL is a declarative language for composing LangChain components. LangChain
compiles LCEL compositions to an optimized execution plan, with automatic paralleli‐
zation, streaming, tracing, and async support.

Let’s see the same example using LCEL:

In [28]:
template = ChatPromptTemplate.from_messages([
('system', 'You are a helpful assistant.'),
('human', '{question}'),
])

model = ChatOllama(model='llama3')

chatbot = template | model

chatbot.invoke({"question": "Which model providers offer LLMs?"})

AIMessage(content="Many natural language processing (NLP) model providers offer large language models (LLMs). Here's a list of some prominent ones:\n\n1. **Hugging Face**: Offers the Transformers library, which provides pre-trained models like BERT, RoBERTa, and XLNet, as well as their own LLaMA model.\n2. **Google**: Provides several LLMs, including T5, BART, and Electra, through their AutoML and TensorFlow repositories.\n3. **Microsoft**: Offers the Azure Cognitive Services Language API, which includes pre-trained models like Turing and Cogito.\n4. **Amazon**: Provides Amazon Comprehend, a natural language processing service that offers LLMs like Amazon Comprehend Custom.\n5. **DeepMind**: Developed the Transformer-XL model and makes it available through their TensorFlow repository.\n6. **Meta AI**: Offers the OPT-175B and CPM-12 models as part of their Meta AI NLP library.\n7. **Stanford University**: Provides the Stanford Question Answering Dataset (SQuAD) and its pre-trained langu